# Pandas Workshop for Neuroscientists

**Dataset:** simulated electrophysiological recordings from neurons across different brain regions, genotypes, and treatment groups.

Goal: after this workshop you'll be able to load your own measurement data, filter it, group it, and compute summary statistics — the everyday operations of working with data.

**Agenda:**
1. Loading & inspecting data
2. Selecting columns, filtering rows
3. Computing new columns
4. **`groupby` — the core of this workshop**
5. Quick outlook: plotting
6. Practice exercises


## 1. Loading & inspecting data

We load neuron measurements: firing rate (Hz) and spine density (dendritic spines per µm), recorded in three brain regions, two genotypes (wild-type vs. knockout of a gene), and three treatment conditions.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('neuron_data.csv')
df.head()

**Column overview:**

| Column | Meaning |
|---|---|
| `neuron_id` | unique ID per recorded neuron |
| `animal_id` | animal the neuron came from (several neurons per animal) |
| `brain_region` | Hippocampus, Prefrontal Cortex, Amygdala |
| `genotype` | WT or Knockout_GeneX |
| `treatment` | Control, Acute_Stress, Deprivation |
| `replicate` | replicate number within the combination |
| `firing_rate_hz` | recorded firing rate in Hz |
| `spine_density` | spine density (a measure of synaptic connectivity) |
| `cell_survived` | 1 = cell survived the recording, 0 = did not |


In [ ]:
# How many rows/columns do we have?
df.shape

In [ ]:
# Data types and missing values at a glance
df.info()

In [ ]:
# Descriptive statistics for all numeric columns
df.describe()

**Note:** `spine_density` has fewer entries than the other columns (`.info()` shows this) — these are real missing measurements (NaN). This happens constantly in real datasets. We'll touch on it again below.


## 2. Selecting columns, filtering rows

### Selecting a single column


In [ ]:
df['firing_rate_hz']

The result is no longer a table but a **Series** (a single column with an index).

In [ ]:
# Selecting multiple columns -> result stays a DataFrame
df[['brain_region', 'firing_rate_hz']]

### Filtering rows with conditions

Probably the single most useful everyday operation: "show me only the rows that meet a certain condition."


In [ ]:
# Only neurons with a high firing rate
df[df['firing_rate_hz'] > 8]

In [ ]:
# Only neurons from the Hippocampus
df[df['brain_region'] == 'Hippocampus']

### Combining multiple conditions

- `&` = AND
- `|` = OR
- Each individual condition **must be wrapped in parentheses**!


In [ ]:
# Hippocampus AND knockout genotype
df[(df['brain_region'] == 'Hippocampus') & (df['genotype'] == 'Knockout_GeneX')]

In [ ]:
# Control OR Deprivation
df[(df['treatment'] == 'Control') | (df['treatment'] == 'Deprivation')]

### `.loc[]` — selecting rows and columns at the same time

`df.loc[row_condition, column_selection]`


In [ ]:
# Only ID and firing rate of neurons with firing rate > 8
df.loc[df['firing_rate_hz'] > 8, ['neuron_id', 'firing_rate_hz']]

## 3. Computing new columns

New columns are created simply through assignment — no special command needed.


In [ ]:
# Example: firing rate per unit of spine density (a made-up metric)
df['firing_rate_per_spine'] = df['firing_rate_hz'] / df['spine_density']
df.head()

In [ ]:
# Summary statistics over a whole column
print('Mean firing rate:', df['firing_rate_hz'].mean())
print('Max firing rate:', df['firing_rate_hz'].max())
print('Standard deviation:', df['firing_rate_hz'].std())

## 4. `groupby` — grouped summaries

This is by far the most useful tool in Pandas for everyday lab work: **Split – Apply – Combine**

1. **Split**: the table is broken into groups based on a column (e.g. `treatment`)
2. **Apply**: a function is applied to each group (e.g. mean)
3. **Combine**: the results are combined back into one table

This is exactly what you do in your head when you say: *"I take all the control animals, compute the mean firing rate, then all the stress animals, compute the mean, ..."* — `groupby` automates that.


In [ ]:
# What happens if we call groupby WITHOUT any aggregation?
df.groupby('treatment')

This isn't a result yet, it's a **GroupBy object** — it's "waiting" for us to tell it what to do with each group (mean? sum? count?).


### a) Simplest example: mean firing rate per treatment

In [ ]:
df.groupby('treatment')['firing_rate_hz'].mean()

Same syntax works for other statistics:

In [ ]:
print(df.groupby('treatment')['firing_rate_hz'].median())
print()
print(df.groupby('treatment')['firing_rate_hz'].std())
print()
print(df.groupby('treatment')['firing_rate_hz'].count())

### b) `.size()` vs `.count()` — how many samples per group?

Important to check **before** doing any statistics: are the groups even the same size?


In [ ]:
# .size() counts all rows per group (including NaNs)
df.groupby('treatment').size()

In [ ]:
# .count() counts only non-missing values per column
# -> here you can see e.g. where spine_density has NaNs
df.groupby('treatment')['spine_density'].count()

### c) Aggregating multiple columns at once


In [ ]:
df.groupby('treatment')[['firing_rate_hz', 'spine_density']].mean()

### d) Different aggregations per column with `.agg()`

Sometimes you want, say, the mean firing rate but the maximum spine density.


In [ ]:
df.groupby('treatment').agg({
    'firing_rate_hz': 'mean',
    'spine_density': 'max',
    'cell_survived': 'sum'   # sum of 0/1 = number of surviving cells
})

### e) Grouping by **multiple** columns at once

This is the case you'll need most often in practice: *"mean firing rate per brain region AND genotype"*.


In [ ]:
df.groupby(['brain_region', 'genotype'])['firing_rate_hz'].mean()

The result has a **MultiIndex** (two levels: brain region + genotype). For further processing (e.g. plotting or saving as CSV) it's often more convenient to turn it back into regular columns:


In [ ]:
result = df.groupby(['brain_region', 'genotype'])['firing_rate_hz'].mean().reset_index()
result

### f) Combination: grouping by 3 factors

Exactly what you'd need for a real 3-factor analysis (region × genotype × treatment):


In [ ]:
df.groupby(['brain_region', 'genotype', 'treatment'])['firing_rate_hz'].agg(['mean', 'std', 'count']).reset_index()

### g) Sorting grouped results

`groupby` combines seamlessly with `.sort_values()`.


In [ ]:
df.groupby('brain_region')['firing_rate_hz'].mean().sort_values(ascending=False)

## 5. Quick outlook: plotting straight away

A `groupby` result can be plotted with no extra steps — handy for a quick first look.


In [ ]:
%matplotlib inline
df.groupby('treatment')['firing_rate_hz'].mean().plot(kind='bar', title='Mean firing rate per treatment')

## 6. Practice exercises

Try the following exercises yourselves — use `df`. Solutions are below each one (only check after you've tried!).

### Exercise 1
Compute the **mean spine density per genotype**.


In [ ]:
# Your code here


<details>
<summary>Show solution</summary>

```python
df.groupby('genotype')['spine_density'].mean()
```
</details>


### Exercise 2
First filter for rows where the cell **survived** (`cell_survived == 1`), then compute the mean firing rate per brain region — sorted in descending order.


In [ ]:
# Your code here


<details>
<summary>Show solution</summary>

```python
df_survived = df[df['cell_survived'] == 1]
df_survived.groupby('brain_region')['firing_rate_hz'].mean().sort_values(ascending=False)
```
</details>


### Exercise 3 (a bit more advanced)
For every combination of `genotype` and `treatment`, compute both the mean and the count of neurons for firing rate. Use `.reset_index()` so the result becomes a regular table.


In [ ]:
# Your code here


<details>
<summary>Show solution</summary>

```python
df.groupby(['genotype', 'treatment'])['firing_rate_hz'].agg(['mean', 'count']).reset_index()
```
</details>


## Summary

| Task | Command |
|---|---|
| Load data | `pd.read_csv()` |
| Get an overview | `.head()`, `.info()`, `.describe()` |
| Select a column | `df['column']` |
| Filter rows | `df[df['column'] > value]` |
| Compute a new column | `df['new'] = ...` |
| Group & aggregate | `df.groupby('column')['value'].mean()` |
| Multiple aggregations | `.agg(['mean', 'std', 'count'])` |
| Flatten a MultiIndex | `.reset_index()` |

**Next steps:** `merge()` (combining multiple tables, e.g. per-animal metadata), `pivot_table()` (cross-tabulations), and connecting directly to `matplotlib`/`seaborn` for publication-ready plots.
